# Silver Transactions (`pipelines/02_silver_transactions.ipynb`)

This notebook turns raw ingested Bronze records (`workspace.sentinel_bronze.transactions`) into two curated Silver tables:
- **`workspace.sentinel_silver.transactions`**: Clean, validated, unique transactions (`record_disposition == "accepted"`) with personal cardholder attributes removed.
- **`workspace.sentinel_silver.quarantine`**: Rejected or duplicate records (`record_disposition != "accepted"`) with raw strings, source lineage, and validation reasons preserved for investigation.

**Pipeline flow**: `read → convert types → validate → classify duplicates → split outputs → save → basic verification`

In [ ]:
# Configuration: imports, table names, lineage/personal column sets, and output column lists
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_TABLE = "workspace.sentinel_bronze.transactions"
SILVER_SCHEMA = "workspace.sentinel_silver"
SILVER_TRANSACTIONS_TABLE = f"{SILVER_SCHEMA}.transactions"
SILVER_QUARANTINE_TABLE = f"{SILVER_SCHEMA}.quarantine"
MANIFEST_PATH = "/Volumes/workspace/sentinel_sources/replay_staging/replay_v1/manifest"

# Lineage metadata columns excluded when building the business-record duplicate signature
LINEAGE_COLUMNS = {
    "source_file",
    "source_row_index",
    "replay_batch",
    "_rescued_data",
    "_ingested_file_path",
    "_ingestion_timestamp",
}

# Personal cardholder attributes excluded from accepted Silver transactions
EXCLUDED_PERSONAL_COLUMNS = {
    "first", "last", "gender", "street", "city", "state",
    "zip", "lat", "long", "city_pop", "job", "dob",
}

ACCEPTED_SILVER_COLUMNS = [
    "trans_num", "cc_num", "merchant", "category", "amount", "event_time",
    "replay_batch", "source_file", "source_row_index",
    "_ingested_file_path", "_ingestion_timestamp",
]

QUARANTINE_SILVER_COLUMNS = [
    "record_disposition", "duplicate_status", "duplicate_rank", "validation_reasons",
    "trans_num", "cc_num", "merchant", "category", "amount", "event_time",
    "raw_trans_num", "raw_cc_num", "raw_merchant", "raw_amt",
    "raw_trans_date_trans_time", "unix_time", "_rescued_data",
    "replay_batch", "source_file", "source_row_index",
    "_ingested_file_path", "_ingestion_timestamp",
]

### Step 1: Read Bronze Transactions

We read `workspace.sentinel_bronze.transactions`, confirm `is_fraud` is absent, and check the total row count.
- **Example**: Each Bronze row stores source fields as `STRING` (e.g., `amt = "42.50"`, `trans_date_trans_time = "2019-01-01 00:01:15"`) plus `replay_batch` (`DATE`) and ingestion metadata.

In [ ]:
# Step 1: Read Bronze transactions and confirm no fraud labels are present
bronze_df = spark.table(BRONZE_TABLE)
assert "is_fraud" not in bronze_df.columns, "Fraud labels must not appear in Bronze"

bronze_input_count = bronze_df.count()
print(f"Read {bronze_input_count:,} Bronze transactions from {BRONZE_TABLE}.")

### Step 2: Convert Types & Normalize Required Identifiers

We keep raw strings (`raw_trans_num`, `raw_amt`, etc.) for quarantine inspection, trim and normalize blank strings to `NULL` via `NULLIF(TRIM(...), '')`, cast `event_time` as `TIMESTAMP_NTZ`, cast `amount` to `DECIMAL(18,2)` alongside `_wide_amount` (`DECIMAL(38,18)`), and build a JSON `_business_signature` across all business columns.
- **Example**:
  - `"  TX_1  "` -> `trans_num = "TX_1"`, while `"   "` -> `NULL`.
  - `amt = "19.999"` -> `_wide_amount = 19.999`, `amount = 20.00` (so Step 3 can detect that 2-decimal rounding changed the value).

In [ ]:
# Step 2: Preserve raw strings, normalize blank identifiers to NULL, cast types, and build _business_signature
business_columns = sorted([
    col_name for col_name in bronze_df.columns if col_name not in LINEAGE_COLUMNS
])

business_signature_struct = F.struct(*[
    F.coalesce(
        F.expr(f"NULLIF(TRIM(`{col_name}`), '')").cast("string"),
        F.lit("__NULL__"),
    ).alias(col_name)
    for col_name in business_columns
])

typed_df = (
    bronze_df
    .withColumn("raw_trans_num", F.col("trans_num"))
    .withColumn("raw_cc_num", F.col("cc_num"))
    .withColumn("raw_merchant", F.col("merchant"))
    .withColumn("raw_amt", F.col("amt"))
    .withColumn("raw_trans_date_trans_time", F.col("trans_date_trans_time"))
    .withColumn("trans_num", F.expr("NULLIF(TRIM(trans_num), '')"))
    .withColumn("cc_num", F.expr("NULLIF(TRIM(cc_num), '')"))
    .withColumn("merchant", F.expr("NULLIF(TRIM(merchant), '')"))
    .withColumn("category", F.expr("NULLIF(TRIM(category), '')"))
    .withColumn("_normalized_event_time_str", F.expr("NULLIF(TRIM(trans_date_trans_time), '')"))
    .withColumn("event_time", F.expr("TRY_CAST(_normalized_event_time_str AS TIMESTAMP_NTZ)"))
    .withColumn("_normalized_amt_str", F.expr("NULLIF(TRIM(amt), '')"))
    .withColumn("_wide_amount", F.expr("TRY_CAST(_normalized_amt_str AS DECIMAL(38, 18))"))
    .withColumn("amount", F.expr("TRY_CAST(_normalized_amt_str AS DECIMAL(18, 2))"))
    .withColumn("_business_signature", F.to_json(business_signature_struct))
)

### Step 3: Validate Required Fields, Amounts, Timestamps & Rescued Data

We collect all applicable field-level validation failures into an array (`field_validation_reasons`) using `F.array(...)` and `F.filter(..., lambda r: r.isNotNull())`.
- **Example**: A row with `amt = "-0.004"` triggers both `"amount_precision_changed"` and `"non_positive_amount"` -> `field_validation_reasons = ["amount_precision_changed", "non_positive_amount"]`.

In [ ]:
# Step 3: Attach all applicable field validation reasons to each row
raw_validation_reasons = F.array(
    F.when(F.col("trans_num").isNull(), F.lit("missing_transaction_id")),
    F.when(F.col("cc_num").isNull(), F.lit("missing_card_id")),
    F.when(F.col("merchant").isNull(), F.lit("missing_merchant")),
    F.when(F.col("_normalized_event_time_str").isNull(), F.lit("missing_event_time")),
    F.when(
        F.col("_normalized_event_time_str").isNotNull() & F.col("event_time").isNull(),
        F.lit("unparseable_event_time"),
    ),
    F.when(F.col("_normalized_amt_str").isNull(), F.lit("missing_amount")),
    F.when(
        F.col("_normalized_amt_str").isNotNull()
        & (F.col("_wide_amount").isNull() | F.col("amount").isNull()),
        F.lit("unparseable_or_out_of_range_amount"),
    ),
    F.when(
        F.col("_wide_amount").isNotNull()
        & F.col("amount").isNotNull()
        & (F.col("_wide_amount") != F.col("amount").cast("decimal(38,18)")),
        F.lit("amount_precision_changed"),
    ),
    F.when(
        F.col("_wide_amount").isNotNull()
        & (F.col("_wide_amount") <= F.lit(0).cast("decimal(38,18)")),
        F.lit("non_positive_amount"),
    ),
    F.when(
        F.col("_rescued_data").isNotNull() & (F.trim(F.col("_rescued_data")) != ""),
        F.lit("unexpected_rescued_data"),
    ),
)

validated_df = typed_df.withColumn(
    "field_validation_reasons",
    F.filter(raw_validation_reasons, lambda reason: reason.isNotNull()),
)

### Step 4: Classify Duplicate Transaction IDs Across All Bronze Rows

We group by non-null `trans_num` across the entire Bronze input (including invalid rows so corrupted rows cannot hide a conflict) to count occurrences and distinct `_business_signature` values:
- **Identical duplicates (`distinct_business_signatures == 1`)**: Rank `1` is `identical_duplicate_primary` (accepted if valid); rank `> 1` is `identical_duplicate_extra` with reason `duplicate_identical_transaction_id`.
- **Conflicting duplicates (`distinct_business_signatures > 1`)**: We never pick an arbitrary winner; every row for that `trans_num` gets `conflicting_duplicate_transaction_id` and is quarantined.
- **Example**: Two rows with `trans_num = "TX_9"` and different amounts (`25.00` vs `99.00`) both become `record_disposition = "quarantined"`.

In [ ]:
# Step 4: Classify duplicate transaction IDs and assign mutually exclusive record_disposition
duplicate_summary = (
    validated_df.filter(F.col("trans_num").isNotNull())
    .groupBy("trans_num")
    .agg(
        F.count("*").alias("id_occurrence_count"),
        F.countDistinct("_business_signature").alias("distinct_business_signatures"),
    )
)

id_order_window = Window.partitionBy("trans_num").orderBy(
    F.col("_ingestion_timestamp").asc_nulls_last(),
    F.col("_ingested_file_path").asc_nulls_last(),
    F.expr("TRY_CAST(source_row_index AS BIGINT)").asc_nulls_last(),
    F.col("source_row_index").asc_nulls_last(),
)

classified_df = (
    validated_df.join(duplicate_summary, on="trans_num", how="left")
    .withColumn(
        "duplicate_rank",
        F.when(F.col("trans_num").isNotNull(), F.row_number().over(id_order_window)).cast("int"),
    )
    .withColumn(
        "duplicate_status",
        F.when(F.col("trans_num").isNull(), F.lit("missing_id"))
        .when(F.col("id_occurrence_count") == 1, F.lit("unique_id"))
        .when(
            (F.col("id_occurrence_count") > 1) & (F.col("distinct_business_signatures") > 1),
            F.lit("conflicting_duplicate"),
        )
        .when(
            (F.col("id_occurrence_count") > 1)
            & (F.col("distinct_business_signatures") == 1)
            & (F.col("duplicate_rank") == 1),
            F.lit("identical_duplicate_primary"),
        )
        .otherwise(F.lit("identical_duplicate_extra")),
    )
    .withColumn(
        "duplicate_validation_reasons",
        F.when(
            F.col("duplicate_status") == "conflicting_duplicate",
            F.array(F.lit("conflicting_duplicate_transaction_id")),
        )
        .when(
            F.col("duplicate_status") == "identical_duplicate_extra",
            F.array(F.lit("duplicate_identical_transaction_id")),
        )
        .otherwise(F.array().cast("array<string>")),
    )
    .withColumn(
        "validation_reasons",
        F.concat(F.col("field_validation_reasons"), F.col("duplicate_validation_reasons")),
    )
    .withColumn(
        "record_disposition",
        F.when(F.size(F.col("validation_reasons")) == 0, F.lit("accepted"))
        .when(
            (F.col("duplicate_status") == "identical_duplicate_extra")
            & (F.size(F.col("field_validation_reasons")) == 0),
            F.lit("duplicate_identical"),
        )
        .otherwise(F.lit("quarantined")),
    )
)

### Step 5: Split Accepted and Quarantine Outputs

Every input row receives exactly one `record_disposition` (`accepted`, `quarantined`, or `duplicate_identical`).
- **`accepted_silver_df`**: `record_disposition == "accepted"`, selecting only `ACCEPTED_SILVER_COLUMNS` (personal cardholder fields excluded).
- **`quarantine_silver_df`**: `record_disposition != "accepted"` (both `quarantined` and `duplicate_identical`), selecting `QUARANTINE_SILVER_COLUMNS`.

In [ ]:
# Step 5: Split classified rows into accepted Silver and quarantine projections
assert EXCLUDED_PERSONAL_COLUMNS.isdisjoint(ACCEPTED_SILVER_COLUMNS), (
    "Personal attributes must not appear in accepted Silver columns"
)

accepted_silver_df = (
    classified_df.filter(F.col("record_disposition") == "accepted")
    .select(*ACCEPTED_SILVER_COLUMNS)
)

quarantine_silver_df = (
    classified_df.filter(F.col("record_disposition") != "accepted")
    .select(*QUARANTINE_SILVER_COLUMNS)
)

disposition_counts = classified_df.agg(
    F.count("*").alias("classified_rows"),
    F.count_if(F.col("record_disposition") == "accepted").alias("accepted_rows"),
    F.count_if(F.col("record_disposition") == "quarantined").alias("quarantined_rows"),
    F.count_if(F.col("record_disposition") == "duplicate_identical").alias("duplicate_identical_rows"),
).collect()[0]

assert disposition_counts["classified_rows"] == bronze_input_count, "Row count mismatch after classification"
assert (
    disposition_counts["accepted_rows"]
    + disposition_counts["quarantined_rows"]
    + disposition_counts["duplicate_identical_rows"]
    == bronze_input_count
), "Disposition counts do not sum to Bronze input count"

print(
    f"Split {bronze_input_count:,} Bronze rows -> "
    f"{disposition_counts['accepted_rows']:,} accepted, "
    f"{disposition_counts['quarantined_rows']:,} quarantined, "
    f"{disposition_counts['duplicate_identical_rows']:,} duplicate_identical."
)

### Step 6: Save Silver and Quarantine Tables (`overwrite` Mode)

We rebuild both `workspace.sentinel_silver.transactions` and `workspace.sentinel_silver.quarantine` from the complete current Bronze snapshot using `.mode("overwrite")`.
- **Why full recomputation**: If a later batch introduces a conflicting duplicate `trans_num`, reclassifying the full Bronze snapshot removes the earlier record from `transactions` and routes all occurrences to `quarantine`.
- **Note**: Each `saveAsTable` is a separate Delta transaction; always run both this cell and Step 7 verification to completion.

In [ ]:
# Step 6: Overwrite workspace.sentinel_silver.transactions and workspace.sentinel_silver.quarantine
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

accepted_silver_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(SILVER_TRANSACTIONS_TABLE)

quarantine_silver_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(SILVER_QUARANTINE_TABLE)

print(f"Saved {SILVER_TRANSACTIONS_TABLE} and {SILVER_QUARANTINE_TABLE} in full-snapshot overwrite mode.")

### Step 7: Basic Silver & Quarantine Verification

We read both persisted tables back from Delta storage and verify:
1. Column lists match `ACCEPTED_SILVER_COLUMNS` and `QUARANTINE_SILVER_COLUMNS` (with `is_fraud` and personal columns excluded).
2. Total and daily batch counts (`saved_accepted + saved_quarantine`) reconcile with Bronze and `/Volumes/workspace/sentinel_sources/replay_staging/replay_v1/manifest`.
3. Accepted `trans_num` values are non-null and unique.

In [ ]:
# Step 7: Read back saved Silver tables and verify counts, manifest reconciliation, and unique accepted IDs
saved_silver_df = spark.table(SILVER_TRANSACTIONS_TABLE)
saved_quarantine_df = spark.table(SILVER_QUARANTINE_TABLE)
manifest_df = spark.read.parquet(MANIFEST_PATH)

assert saved_silver_df.columns == ACCEPTED_SILVER_COLUMNS, "Unexpected columns in Silver transactions"
assert saved_quarantine_df.columns == QUARANTINE_SILVER_COLUMNS, "Unexpected columns in Silver quarantine"
assert "is_fraud" not in saved_silver_df.columns and "is_fraud" not in saved_quarantine_df.columns

saved_accepted_count = saved_silver_df.count()
saved_quarantine_count = saved_quarantine_df.count()
expected_quarantine_count = (
    disposition_counts["quarantined_rows"] + disposition_counts["duplicate_identical_rows"]
)

assert saved_accepted_count == disposition_counts["accepted_rows"], "Saved accepted count mismatch"
assert saved_quarantine_count == expected_quarantine_count, "Saved quarantine count mismatch"
assert saved_accepted_count + saved_quarantine_count == bronze_input_count, "Total saved rows != Bronze count"

null_accepted_ids = saved_silver_df.filter(F.col("trans_num").isNull()).count()
distinct_accepted_ids = saved_silver_df.select("trans_num").distinct().count()
assert null_accepted_ids == 0, "NULL trans_num found in accepted Silver"
assert distinct_accepted_ids == saved_accepted_count, "Duplicate trans_num found in accepted Silver"

# Reconcile daily counts against the replay manifest for all ingested batches.
accepted_by_batch = saved_silver_df.groupBy("replay_batch").agg(
    F.count("*").alias("saved_accepted_rows"),
    F.countDistinct("trans_num").alias("distinct_accepted_ids"),
)
quarantine_by_batch = saved_quarantine_df.groupBy("replay_batch").agg(
    F.count("*").alias("saved_quarantine_rows"),
)
ingested_batches = bronze_df.select("replay_batch").distinct()

batch_reconciliation = (
    manifest_df.join(ingested_batches, on="replay_batch", how="inner")
    .join(accepted_by_batch, on="replay_batch", how="left")
    .join(quarantine_by_batch, on="replay_batch", how="left")
    .fillna(0, subset=["saved_accepted_rows", "distinct_accepted_ids", "saved_quarantine_rows"])
    .withColumn("saved_total_rows", F.col("saved_accepted_rows") + F.col("saved_quarantine_rows"))
    .withColumn("matches_manifest", F.col("saved_total_rows") == F.col("expected_transaction_count"))
    .orderBy("replay_batch")
)

assert batch_reconciliation.filter(~F.col("matches_manifest")).count() == 0, (
    "Daily Silver + quarantine counts do not match the manifest"
)

print(
    f"Verified {SILVER_TRANSACTIONS_TABLE}: {saved_accepted_count:,} rows "
    f"({distinct_accepted_ids:,} unique trans_num)."
)
print(f"Verified {SILVER_QUARANTINE_TABLE}: {saved_quarantine_count:,} rows.")
display(batch_reconciliation)